# NeuroVascNet v4 — EEG → hemodynamics (Kaggle)

**Status:** current. The first uncontaminated test of the thesis.

## Purpose

One shared EEG encoder, trained with paired EEG–fNIRS *and* EEG–fMRI supervision, that at
deployment takes **EEG only** and predicts the haemodynamic response. This notebook is the
results section: it re-runs the evaluation after a data-quality audit showed every previous
result was confounded by broken supervision targets.

## What we did

* Replaced **HEFMI** with **TU Berlin hBCI** as the primary fNIRS training set, reprocessed
  from raw 760/850 nm intensities with our own Beer–Lambert conversion.
* Turned **off** global-signal regression on the target.
* Rebuilt the **fMRI** target: event-locked on stimulus onsets via the R128 scanner
  triggers, motion + drift regressed out, restricted to sensorimotor ROIs.
* Gated every target on **split-half reliability** before believing any prediction.
* Scored everything against a **constant-predictor baseline** and a **permutation null**.

## What we achieved

| target | reliability | ceiling | verdict |
|---|---|---|---|
| fNIRS HEFMI (old) | +0.207 | 0.455 | HbO/HbR **+0.898** — physiologically wrong |
| **fNIRS hBCI (new)** | **+0.440** | 0.663 | correct sign, true SCI 0.902 |
| fMRI v3 (old) | **−0.639** | — | does not reproduce; all its numbers spurious |
| **fMRI v4 (rebuilt)** | **+0.638** | 0.799 | best target in the project |

* **Both arms now have targets that reproduce** — the first time in this project.
* **A textbook false positive, caught and dissolved.** The broken fMRI target gave
  r = 0.0706 at **p = 0.005**; after repair the same pipeline gives **p = 0.473**.
* **A real cross-subject EEG contribution:** subject-demeaned **r = 0.053 (p = 0.005)**,
  rising to **0.071** when three levers are combined (delay alignment + ERD features +
  extra training data). The constant baseline is 0 by construction, so this is purely EEG.

## What we failed to achieve

* **The effect is small** — 0.071 against a ceiling of 0.663, roughly 11%.
* **The raw metric is misleading.** Plain concat-Pearson gives 0.072, but an EEG-free
  constant predictor scores **0.076** — it beats the model. Trial averaging widens that
  gap (0.243 vs 0.309 at k=30), so a bigger raw number is a *worse* result. Judge on
  `r_demeaned` only.
* **Amplitude did not survive** the move to the repaired data (p ≥ 0.07 everywhere).
* **The deep encoder did not beat ridge** (0.054 vs 0.072 raw). Its loss was still falling
  at 60 epochs, so it is likely underfit rather than saturated — one data point, not a
  settled result.
* **fMRI has no positive result at any setting**, and with 263 training trials versus 1380
  for fNIRS it is underpowered rather than disproven.
* Reliability bounds what is *achievable*, not what is *reachable*: a response can be
  perfectly reproducible while its trial-to-trial variation is driven by vascular and
  systemic factors EEG cannot see. That, rather than model capacity, is the most likely
  explanation for the remaining gap.

## Setup

1. **Settings → Accelerator → GPU** (T4 or P100)
2. **Settings → Internet → On** — required, the notebook downloads its own data
3. Run all cells.

Everything is built from scratch from the Google Drive folders: download → Beer–Lambert
conversion → epoching → caches → analysis. Nothing needs to be uploaded.

| stage | approx. time |
|---|---|
| download hBCI (6.6 GB) + ds002336 (18 GB) | 30–90 min |
| build fNIRS cache (29 subjects, raw 760/850 nm) | ~10 min |
| build fMRI cache (ROI extraction + event-locking) | ~30 min |
| analysis + deep model | ~10 min |

Comfortably inside a 12-hour session. Caches are written to `/kaggle/working`, so
re-running skips any stage already completed.

**Disk:** ~25 GB of raw download plus ~2 GB of caches. Set `DELETE_RAW_AFTER_CACHE = True`
in the download cell to free the raw data once the caches exist.

## What it reports

Three numbers for every result, because the first one alone is misleading:

| metric | meaning |
|---|---|
| `r_raw` | plain concat-Pearson — **inflated** by the stereotyped HRF shape shared by all trials |
| `r_const` | what you get **ignoring EEG entirely** and emitting the training mean |
| `r_demeaned` | subject means removed, so `r_const` collapses to 0 — **this is the real EEG contribution** |

Judge on `r_demeaned`. On the reference run, `r_raw = 0.072` looked significant but the
constant predictor scored `0.076`; the honest figure is `r_demeaned = 0.053` (p = 0.005),
rising to **0.071** with the three levers combined.

In [1]:
import os, sys, glob, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

ON_KAGGLE = Path("/kaggle").exists()
print("running on Kaggle:", ON_KAGGLE)

try:
    import torch
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch", torch.__version__, "| device:", DEV,
          "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "")
except Exception as e:
    DEV = "cpu"; print("torch unavailable:", e)

running on Kaggle: False
torch 2.7.1+cu118 | device: cuda | NVIDIA GeForce RTX 3060 Ti


## 1. Download the raw datasets from Google Drive

`gdown.download_folder` is limited to 50 files per folder unless `remaining_ok=True` is
passed — these folders contain hundreds, so that flag is essential. If Drive rate-limits
the pull (it does happen on multi-GB folders), re-run this cell: completed files are
skipped, so it resumes rather than restarting.

In [2]:
RUN_DOWNLOAD = True
DELETE_RAW_AFTER_CACHE = False     # True frees ~25 GB once the caches are built

DRIVE_FOLDERS = {
    "TU Berlin hBCI EEG+FNRI":  "1lw-AR79e5FgVOL6iHaRG0FppYJHOhnqj",   # 6.6 GB  fNIRS arm
    "OpenNeuro ds002336":       "1K_91Y5Ph0PiAIcFdlGsiHg10_knUNmhw",   # 18 GB   fMRI arm
    # Not needed for the headline result. HEFMI is the superseded fNIRS set,
    # TU Berlin WL and ds002739 are external validation sets this notebook does not use.
    # "HEFMI-ICH EEG+FNRI":     "1OnC-Z62tLH8ompyxj4ev_qABiqIkj8jv",   # 13 GB
    # "TU Berlin EEG + NIRS":   "166XH-wTGQy7RqPbgE9uUn-EpsTLp3HIF",   # 23 GB
    # "OpenNeuro ds002739":     "1Rss12hlZrLqQTJDP3GtR_MqIwyDWNy1N",   # 43 GB
}

WORK = Path("/kaggle/working" if ON_KAGGLE else ".")
RAW = WORK / "dataset"; RAW.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "cache"; CACHE.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--upgrade", "gdown", "mne", "nilearn"], check=False)

if RUN_DOWNLOAD:
    import gdown
    for name, fid in DRIVE_FOLDERS.items():
        dest = RAW / name
        n_before = len(list(dest.rglob("*"))) if dest.exists() else 0
        print(f"\n--- {name}  ({n_before} files present)")
        try:
            gdown.download_folder(id=fid, output=str(dest), quiet=False,
                                  use_cookies=False, remaining_ok=True)
        except Exception as e:
            print(f"  download issue: {type(e).__name__}: {e}")
            print("  re-run this cell to resume (finished files are skipped)")
        print(f"  now {len(list(dest.rglob('*')))} files")

for name in DRIVE_FOLDERS:
    d = RAW / name
    n = len(list(d.rglob("*"))) if d.exists() else 0
    print(f"{name:28s} {n:5d} files")


--- TU Berlin hBCI EEG+FNRI  (440 files present)
  download issue: TypeError: download_folder() got an unexpected keyword argument 'remaining_ok'
  re-run this cell to resume (finished files are skipped)
  now 440 files

--- OpenNeuro ds002336  (755 files present)
  download issue: TypeError: download_folder() got an unexpected keyword argument 'remaining_ok'
  re-run this cell to resume (finished files are skipped)
  now 755 files
TU Berlin hBCI EEG+FNRI        440 files
OpenNeuro ds002336             755 files


## 2a. Build the fNIRS cache — TU Berlin hBCI, from raw wavelengths

The dataset ships **raw 760/850 nm intensities** (36 source–detector pairs × 2 wavelengths),
so the Beer–Lambert conversion is done here with known constants rather than trusting a
vendor pipeline. That matters: the superseded HEFMI set had HbO/HbR = **+0.898**, which is
physiologically impossible for a real response and is what a bad conversion looks like.

EEG and NIRS runs are paired within subject and run. Only the **MI** (motor-imagery) runs
feed the main result; **MA** (mental arithmetic) is built too, for Lever 3.

In [ ]:
import scipy.io as sio
from scipy import signal as sps

HBCI = RAW / "TU Berlin hBCI EEG+FNRI"
EXT = {760: {"HbO": 1486.59, "HbR": 3843.71}, 850: {"HbO": 2526.39, "HbR": 1798.64}}
DPF = {760: 6.4, 850: 5.8}
SD_CM, EOG = 3.0, ("VEOG", "HEOG")
TMIN, TMAX, EEG_SAMPLES, HEMO_SAMPLES = -2.0, 28.0, 6000, 300

def _runs(p, hint=None):
    m = sio.loadmat(p, squeeze_me=True, struct_as_record=False)
    keys = [k for k in m if not k.startswith("__")]
    return np.atleast_1d(m[hint if hint in keys else keys[0]]).ravel()

def _bp(x, fs, lo, hi, order=3):
    nyq = fs/2; hi = min(hi, nyq*0.95)
    if lo >= hi: return x
    b, a = sps.butter(order, [lo/nyq, hi/nyq], btype="band")
    return sps.filtfilt(b, a, x, axis=-1)

def beer_lambert(lo_wl, hi_wl):
    def od(x):
        base = np.median(x, axis=-1, keepdims=True)
        base = np.where(np.abs(base) < 1e-12, 1e-12, base)
        return -np.log(np.clip(x/base, 1e-9, None))
    a1, a2 = od(lo_wl), od(hi_wl)
    L1, L2 = SD_CM*DPF[760], SD_CM*DPF[850]
    E = np.array([[EXT[760]["HbO"]*L1, EXT[760]["HbR"]*L1],
                  [EXT[850]["HbO"]*L2, EXT[850]["HbR"]*L2]])
    Ei = np.linalg.inv(E)
    return (Ei[0,0]*a1 + Ei[0,1]*a2)*1e6, (Ei[1,0]*a1 + Ei[1,1]*a2)*1e6

def split_wavelengths(x, clab):
    names = [str(c) for c in np.atleast_1d(clab)]
    lo = {n.replace("lowWL",""): i for i,n in enumerate(names) if "lowWL" in n}
    hi = {n.replace("highWL",""): i for i,n in enumerate(names) if "highWL" in n}
    pairs = sorted(set(lo) & set(hi))
    return (np.stack([x[:,lo[p]] for p in pairs]), np.stack([x[:,hi[p]] for p in pairs]))

def _eeg_dir(n):
    for g in sorted((HBCI/"EEG data").glob("EEG_*")):
        d = g / f"subject {n:02d}" / "with occular artifact"
        if d.exists(): return d
    return None

def _rs(x, n): return x if x.shape[-1] == n else sps.resample(x, n, axis=-1)

def build_hbci(task="MI", n_val=6):
    out_tr = CACHE/f"hbci_train_{task}.npz"; out_va = CACHE/f"hbci_val_{task}.npz"
    if out_tr.exists() and out_va.exists():
        print(f"hbci {task}: cached"); return
    subs = list(range(1, 30)); val = set(subs[-n_val:])
    acc = {"train": [[],[],[],[]], "val": [[],[],[],[]]}
    nb = int(2.0*10)
    for s in subs:
        ed, nd = _eeg_dir(s), HBCI/"NRI data"/"NIRS_01-29"/f"subject {s:02d}"
        if ed is None or not nd.exists(): print(f"  subject {s:02d}: missing"); continue
        split = "val" if s in val else "train"
        try:
            er, em = _runs(ed/"cnt.mat","cnt"), _runs(ed/"mrk.mat","mrk")
            nr, nm = _runs(nd/"cnt.mat","cnt"), _runs(nd/"mrk.mat","mrk")
        except Exception as e:
            print(f"  subject {s:02d}: {e}"); continue
        cnt = 0
        for r in range(min(er.size, nr.size)):
            E, N = er[r], nr[r]
            if not str(getattr(E,"title","")).upper().startswith(task): continue
            efs = float(np.atleast_1d(E.fs).ravel()[0])
            clab = [str(c) for c in np.atleast_1d(E.clab)]
            keep = [i for i,c in enumerate(clab) if c not in EOG]
            ex = np.asarray(E.x, dtype=np.float64)[:, keep].T
            nfs = float(np.atleast_1d(N.fs).ravel()[0])
            lo, hi = split_wavelengths(np.asarray(N.x, dtype=np.float64), N.clab)
            hbo, hbr = beer_lambert(lo, hi)
            hbo, hbr = _bp(hbo,nfs,0.01,0.2), _bp(hbr,nfs,0.01,0.2)
            if r >= em.size or r >= nm.size: continue
            eon = np.atleast_1d(np.asarray(em[r].time,dtype=float))/1000.0
            non = np.atleast_1d(np.asarray(nm[r].time,dtype=float))/1000.0
            y = np.asarray(em[r].y); lab = np.argmax(y,0) if y.ndim==2 else np.asarray(y).ravel()
            for k in range(min(eon.size, non.size, lab.size)):
                ae,be = int(round((eon[k]+TMIN)*efs)), int(round((eon[k]+TMAX)*efs))
                an,bn = int(round((non[k]+TMIN)*nfs)), int(round((non[k]+TMAX)*nfs))
                if ae<0 or be>ex.shape[1] or an<0 or bn>hbo.shape[1]: continue
                se = _rs(ex[:,ae:be], EEG_SAMPLES)
                se = ((se - se.mean(-1,keepdims=True))/(se.std(-1,keepdims=True)+1e-6)).astype(np.float32)
                so, sr = _rs(hbo[:,an:bn],HEMO_SAMPLES), _rs(hbr[:,an:bn],HEMO_SAMPLES)
                so = so - so[:,:nb].mean(-1,keepdims=True); sr = sr - sr[:,:nb].mean(-1,keepdims=True)
                acc[split][0].append(se)
                acc[split][1].append(np.concatenate([so,sr],0).astype(np.float32))
                acc[split][2].append(int(lab[k])); acc[split][3].append(s); cnt += 1
        print(f"  subject {s:02d} [{split}] {cnt} trials")
    for split, path in (("train",out_tr), ("val",out_va)):
        e,h,l,sid = acc[split]
        if not e: print(f"  {split}: EMPTY"); continue
        np.savez_compressed(path, eeg=np.stack(e), hemo=np.stack(h),
                            labels=np.asarray(l), subject_ids=np.asarray(sid))
        print(f"{path.name}: {len(e)} trials, {len(set(sid))} subjects")

if HBCI.exists():
    build_hbci("MI"); build_hbci("MA")
else:
    print("hBCI raw data not found - run the download cell")

  subject 01 [train] 60 trials
  subject 02 [train] 60 trials
  subject 03 [train] 60 trials
  subject 04 [train] 60 trials
  subject 05 [train] 60 trials
  subject 06 [train] 60 trials
  subject 07 [train] 60 trials


## 2b. Build the fMRI cache — ds002336, event-locked

The v3 fMRI target had split-half reliability of **−0.793**: it did not reproduce against
itself, so every number computed against it was spurious. Three fixes, applied here:

1. **event-locked** on stimulus onsets rather than sliding windows (the decisive one)
2. **motion + polynomial drift regressed out** before epoching
3. **restricted to sensorimotor ROIs** instead of all 48 Harvard–Oxford regions

EEG↔BOLD alignment uses the **R128 scanner triggers** in the BrainVision markers, which mark
every volume acquisition — so volume *k* is EEG sample `R128[k]` exactly, no assumed offset.

In [ ]:
import mne, nibabel as nib
from nilearn.datasets import fetch_atlas_harvard_oxford
from nilearn.image import resample_to_img

DS = RAW / "OpenNeuro ds002336"
TR, EPOCH_VOL, N_ROI = 2.0, 15, 48
MOTOR_KEYS = ("Precentral","Postcentral","Supplementary","Juxtapositional")
FMRI_TASKS = ("motorloc","MIpre","MIpost","eegNF","fmriNF","eegfmriNF")

def roi_series(nii, atlas_img, labels):
    img = nib.load(str(nii)); data = np.asarray(img.dataobj, dtype=np.float32)
    if data.ndim != 4: return None, None
    ref = nib.Nifti1Image(data[...,0], img.affine, img.header)
    lab = np.asarray(resample_to_img(atlas_img, ref, interpolation="nearest",
                                     force_resample=True, copy_header=True).dataobj).astype(int)
    ts = np.zeros((data.shape[3], N_ROI), np.float32)
    for k in range(1, N_ROI+1):
        m = lab == k
        if m.sum() >= 10: ts[:,k-1] = data[m].mean(0)
    return ts, labels

def clean_series(ts, motion):
    n = ts.shape[0]; t = np.linspace(-1,1,n)
    cols = [np.ones(n), t, t**2, t**3]
    if motion is not None and motion.shape[0] >= 2:
        m = motion[:n] if motion.shape[0]>=n else np.vstack([motion, np.repeat(motion[-1:], n-motion.shape[0],0)])
        for k in range(min(6, m.shape[1])):
            v = m[:,k]; sd = v.std(); cols.append((v-v.mean())/(sd if sd>0 else 1.0))
    X = np.column_stack(cols)
    return ts - X @ np.linalg.lstsq(X, ts, rcond=None)[0]

def build_fmri(n_val=3):
    out_tr, out_va = CACHE/"ds002336_fixed_train.npz", CACHE/"ds002336_fixed_val.npz"
    if out_tr.exists() and out_va.exists(): print("fMRI: cached"); return
    atlas = fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
    amap = atlas.maps if hasattr(atlas,"maps") else atlas["maps"]
    alabels = [str(x) for x in (atlas.labels if hasattr(atlas,"labels") else atlas["labels"])[1:N_ROI+1]]
    motor = [i for i,nm in enumerate(alabels) if any(k in nm for k in MOTOR_KEYS)]
    print("motor ROIs:", [alabels[i] for i in motor])

    subs = sorted([d.name for d in DS.glob("sub-*") if d.is_dir()])
    val = set(subs[-n_val:])
    acc = {"train": [[],[],[]], "val": [[],[],[]]}
    for si, sub in enumerate(subs):
        split = "val" if sub in val else "train"; cnt = 0
        for task in FMRI_TASKS:
            nii = DS/sub/"func"/f"{sub}_task-{task}_bold.nii.gz"
            vhdr = DS/"derivatives"/sub/"eeg_pp"/f"{sub}_task-{task}_eeg_pp.vhdr"
            ev = DS/f"task-{task}_events.tsv"
            if not (nii.exists() and vhdr.exists() and ev.exists()): continue
            try:
                ts, _ = roi_series(nii, amap, alabels)
                if ts is None: continue
                hm = DS/"derivatives"/sub/"head_motion"/f"d_{sub}_task-{task}_headmotion.tsv"
                motion = None
                if hm.exists():
                    df = pd.read_csv(hm, sep="\t").select_dtypes("number").dropna(axis=1, how="all")
                    if df.shape[1] >= 6: motion = df.to_numpy()
                ts = clean_series(ts, motion)[:, motor]

                raw = mne.io.read_raw_brainvision(str(vhdr), preload=True, verbose="ERROR")
                eeg = raw.get_data(picks=mne.pick_types(raw.info, eeg=True, exclude=[]))*1e6
                fs = float(raw.info["sfreq"])
                trig = np.array([o for o,d_ in zip(raw.annotations.onset, raw.annotations.description)
                                 if "R128" in str(d_)])
                trig = (trig*fs).astype(int)
                if trig.size < EPOCH_VOL: continue

                e = pd.read_csv(ev, sep="\t")
                if "trial_type" in e.columns:
                    e = e[~e["trial_type"].astype(str).str.lower().str.contains("rest")]
                onsets = pd.to_numeric(e["onset"], errors="coerce").dropna().to_numpy()
                for o in onsets:
                    v = int(round(o/TR))
                    if v < 0 or v+EPOCH_VOL > ts.shape[0] or v >= trig.size: continue
                    s0 = trig[v]
                    if s0+EEG_SAMPLES > eeg.shape[1]: continue
                    sh = ts[v:v+EPOCH_VOL].T; sh = sh - sh[:,:1]
                    se = eeg[:, s0:s0+EEG_SAMPLES]
                    se = (se - se.mean(-1,keepdims=True))/(se.std(-1,keepdims=True)+1e-6)
                    acc[split][0].append(se.astype(np.float32))
                    acc[split][1].append(sh.astype(np.float32))
                    acc[split][2].append(si); cnt += 1
                del eeg, raw
            except Exception as exc:
                print(f"  {sub}/{task}: {type(exc).__name__}: {str(exc)[:70]}")
        print(f"  {sub} [{split}] {cnt} trials")
    for split, path in (("train",out_tr), ("val",out_va)):
        e,h,sid = acc[split]
        if not e: print(f"  {split}: EMPTY"); continue
        nc = min(x.shape[0] for x in e); nr = min(x.shape[0] for x in h)
        np.savez_compressed(path, eeg=np.stack([x[:nc] for x in e]),
                            hemo=np.stack([x[:nr] for x in h]),
                            labels=np.zeros(len(e), np.int64), subject_ids=np.asarray(sid))
        print(f"{path.name}: {len(e)} trials, {len(set(sid))} subjects")

if DS.exists():
    build_fmri()
else:
    print("ds002336 raw data not found - run the download cell")

In [ ]:
FOUND = {p.name: p for p in CACHE.rglob("*.npz")}
for root in ["/kaggle/input"]:
    if Path(root).exists():
        for p in Path(root).rglob("*.npz"): FOUND.setdefault(p.name, p)
print(f"{len(FOUND)} cache files available:")
for n, p in sorted(FOUND.items()):
    print(f"   {n:32s} {p.stat().st_size/1e6:8.1f} MB")

if DELETE_RAW_AFTER_CACHE and {"hbci_train_MI.npz","ds002336_fixed_train.npz"} <= set(FOUND):
    import shutil
    shutil.rmtree(RAW, ignore_errors=True)
    print("\nraw data deleted, caches retained")

## 3. Load the arms

In [ ]:
def load(train_name, val_name, modality, hemo_fs, chromophores):
    if train_name not in FOUND or val_name not in FOUND:
        return None
    tr, va = np.load(FOUND[train_name], allow_pickle=True), np.load(FOUND[val_name], allow_pickle=True)
    return dict(eeg_tr=tr["eeg"], hemo_tr=tr["hemo"].astype(np.float64), sid_tr=tr["subject_ids"],
                eeg_va=va["eeg"], hemo_va=va["hemo"].astype(np.float64), sid_va=va["subject_ids"],
                modality=modality, hemo_fs=hemo_fs, chromophores=chromophores)

SPEC = {
    "fNIRS hBCI (v4)":  ("hbci_train_MI.npz", "hbci_val_MI.npz", "fnirs", 10.0, True),
    "fMRI v4 (fixed)":  ("ds002336_fixed_train.npz", "ds002336_fixed_val.npz", "fmri", 0.5, False),
    # These two are the SUPERSEDED targets. They are only present if you separately
    # attach the v3 caches as a Kaggle Dataset; the download path does not build them.
    "fNIRS HEFMI (v3)": ("fnirs_train.npz", "fnirs_val.npz", "fnirs", 10.0, True),
    "fMRI v3 (broken)": ("ds002336_train.npz", "ds002336_val.npz", "fmri", 0.5, False),
}
DATA = {k: d for k, v in SPEC.items() if (d := load(*v)) is not None}

for k, d in DATA.items():
    print(f"{k:20s} {d['modality']:5s} train {str(d['eeg_tr'].shape):20s}"
          f" {np.unique(d['sid_tr']).size:2d} subj   val {str(d['eeg_va'].shape):19s}"
          f" {np.unique(d['sid_va']).size} subj   hemo {d['hemo_tr'].shape[1]}x{d['hemo_tr'].shape[2]}")
if not DATA:
    raise SystemExit("No datasets loaded - attach the cache files first.")

## 3. Target quality — the ceiling

Split a subject's trials in half, average each half, correlate. A target that cannot predict
*itself* cannot be predicted by any model, so this bounds everything: max achievable
correlation is `sqrt(reliability)`. Computed over **all** subjects, 50 random splits each,
with a bootstrap CI.

In [ ]:
EEG_FS, N_PERM, N_SPLIT, N_BOOT = 200.0, 200, 50, 2000

def concat_r(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 10 or a[ok].std() == 0 or b[ok].std() == 0: return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0,1])

def subject_reliability(t, n_repeat=N_SPLIT, seed=0):
    rng = np.random.default_rng(seed); n = t.shape[0]
    if n < 6: return np.nan
    rs = [concat_r(t[p[:n//2]].mean(0), t[p[n//2:]].mean(0))
          for p in (rng.permutation(n) for _ in range(n_repeat))]
    rs = [r for r in rs if np.isfinite(r)]
    return float(np.mean(rs)) if rs else np.nan

def reliability_all(d):
    hemo = np.concatenate([d["hemo_tr"], d["hemo_va"]])
    sid = np.concatenate([d["sid_tr"], d["sid_va"] + d["sid_tr"].max() + 1])
    vals = np.array([v for v in (subject_reliability(hemo[sid == s]) for s in np.unique(sid))
                     if np.isfinite(v)])
    if vals.size == 0: return np.nan, np.nan, np.nan, 0
    rng = np.random.default_rng(0)
    boot = [rng.choice(vals, vals.size, replace=True).mean() for _ in range(N_BOOT)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(vals.mean()), float(lo), float(hi), int(vals.size)

qual = pd.DataFrame([{"dataset": k, "modality": d["modality"],
                      **dict(zip(["reliability","ci_low","ci_high","n_subjects"], reliability_all(d))),
                      "ceiling": float(np.sqrt(max(reliability_all(d)[0], 0)))}
                     for k, d in DATA.items()])
print(qual.round(3).to_string(index=False))
print("\nA target with reliability <= 0 cannot be genuinely predicted;")
print("any 'significant' correlation against it is spurious.")

## 4. Features, model, and the three metrics

In [ ]:
BANDS_BASE = [(1,4),(4,8),(8,13),(13,30),(30,45)]

def features(eeg, bands=BANDS_BASE, n_bins=6, erd=False):
    N, C, T = eeg.shape; L = T // n_bins
    x = eeg[:, :, :L*n_bins].reshape(N, C, n_bins, L)
    x = x - x.mean(-1, keepdims=True)
    P = np.abs(np.fft.rfft(x * np.hanning(L), axis=-1))**2
    f = np.fft.rfftfreq(L, 1/EEG_FS)
    bp = np.stack([P[..., (f>=lo)&(f<hi)].sum(-1) for lo,hi in bands], -1)
    logbp = np.log10(bp + 1e-12)
    out = [logbp.reshape(N,-1)]
    if erd: out.append((logbp - logbp[:,:,:1,:]).reshape(N,-1))
    return np.concatenate(out,1).astype(np.float32)

def ridge(Xtr, Ytr, Xte, alphas=(1e0,1e1,1e2,1e3,1e4,1e5)):
    mu, sd = Xtr.mean(0), Xtr.std(0)+1e-8
    Xtr, Xte = (Xtr-mu)/sd, (Xte-mu)/sd
    ym = Ytr.mean(0); Yc = Ytr-ym
    n = int(Xtr.shape[0]*0.8); best, best_r = alphas[len(alphas)//2], -np.inf
    if Xtr.shape[0]-n >= 4:
        G = Xtr[:n].T@Xtr[:n]
        for al in alphas:
            W = np.linalg.solve(G+al*np.eye(G.shape[0]), Xtr[:n].T@Yc[:n])
            r = concat_r(Xtr[n:]@W, Yc[n:])
            if np.isfinite(r) and r > best_r: best_r, best = r, al
    G = Xtr.T@Xtr
    return Xte @ np.linalg.solve(G+best*np.eye(G.shape[0]), Xtr.T@Yc) + ym

def perm_null(pred, true, n=N_PERM, seed=0):
    rng = np.random.default_rng(seed); obs = concat_r(pred, true)
    null = np.array([r for r in (concat_r(pred[rng.permutation(len(pred))], true)
                                 for _ in range(n)) if np.isfinite(r)])
    if null.size == 0: return obs, np.nan, np.nan
    return obs, float(null.mean()), float((np.sum(null>=obs)+1)/(null.size+1))

def demean(A, sid):
    out = A.copy()
    for s in np.unique(sid): out[sid==s] -= out[sid==s].mean(0, keepdims=True)
    return out

def score(Xtr, Ytr, Xva, Yva, sva):
    """Return raw r, constant-predictor r, and demeaned r (+ its permutation p)."""
    pred = ridge(Xtr, Ytr, Xva)
    const = np.repeat(Ytr.mean(0)[None,:], len(Yva), axis=0)
    r_raw = concat_r(pred, Yva)
    r_const = concat_r(const, Yva)
    r_dm, null_dm, p_dm = perm_null(demean(pred, sva), demean(Yva, sva))
    return dict(r_raw=r_raw, r_const=r_const, r_gain=r_raw-r_const,
                r_demeaned=r_dm, null=null_dm, p=p_dm)

## 5. Main result — cross-subject, held-out subjects

In [ ]:
rows = []
for k, d in DATA.items():
    Xtr, Xva = features(d["eeg_tr"]), features(d["eeg_va"])
    Ytr = d["hemo_tr"].reshape(len(d["hemo_tr"]), -1)
    Yva = d["hemo_va"].reshape(len(d["hemo_va"]), -1)
    s = score(Xtr, Ytr, Xva, Yva, d["sid_va"])
    rows.append({"dataset": k, **s})
    print(f"{k:20s} raw {s['r_raw']:+.4f}  const {s['r_const']:+.4f}  "
          f"gain {s['r_gain']:+.4f}   DEMEANED {s['r_demeaned']:+.4f} (p={s['p']:.3f})")
main = pd.DataFrame(rows)
print("\n'gain' <= 0 means the model does not beat ignoring EEG. Judge on DEMEANED.")

## 6. Three levers combined

Individually each does almost nothing; together they gave +35% on the reference run.

* **delay** — response window offset chosen on training subjects only (EEG→haemodynamic lag is 2–8 s)
* **ERD** — power relative to the pre-stimulus baseline, which raw band-power discards
* **more data** — mental-arithmetic runs added to training; the test set stays motor-imagery

In [ ]:
d = DATA.get("fNIRS hBCI (v4)")
if d is None:
    print("fNIRS hBCI cache not loaded, skipping")
else:
    Etr, Htr, str_ = d["eeg_tr"], d["hemo_tr"], d["sid_tr"]
    Eva, Hva, sva = d["eeg_va"], d["hemo_va"], d["sid_va"]

    # Lever 3: append MA runs if available
    if "hbci_train_MA.npz" in FOUND:
        m = np.load(FOUND["hbci_train_MA.npz"], allow_pickle=True)
        Etr = np.concatenate([Etr, m["eeg"]]); Htr = np.concatenate([Htr, m["hemo"].astype(np.float64)])
        print(f"Lever 3: training set {len(Htr)} trials (MI + MA)")
    else:
        print("Lever 3: hbci_train_MA.npz not found, using MI only")

    # Lever 2: ERD + finer bins + more bands
    KW = dict(bands=[(1,4),(4,8),(8,10),(10,13),(13,20),(20,30),(30,45),(45,70)],
              n_bins=12, erd=True)
    Xtr, Xva = features(Etr, **KW), features(Eva, **KW)

    # Lever 1: delay-aligned window. The offset is chosen using TRAINING rows only,
    # so no held-out subject influences the choice.
    sid_pool = np.concatenate([str_, str_]) if len(str_) < len(Htr) else str_
    sid_pool = sid_pool[:len(Htr)]
    win, best_off, best_tr = 100, 0, -np.inf
    for off in range(0, 201, 20):
        if off + win > Htr.shape[2]: continue
        y = Htr[:, :, off:off+win].reshape(len(Htr), -1)
        n = int(len(y) * 0.8)
        pred_in = ridge(Xtr[:n], y[:n], Xtr[n:])
        r = concat_r(demean(pred_in, sid_pool[n:]), demean(y[n:], sid_pool[n:]))
        if np.isfinite(r) and r > best_tr:
            best_tr, best_off = r, off
    print(f"Lever 1: window offset {best_off/d['hemo_fs'] - 2:.0f} s after onset "
          f"(train score {best_tr:+.4f})")

    s = score(Xtr, Htr[:, :, best_off:best_off+win].reshape(len(Htr), -1),
              Xva, Hva[:, :, best_off:best_off+win].reshape(len(Hva), -1), sva)
    print(f"\nCOMBINED   raw {s['r_raw']:+.4f}  const {s['r_const']:+.4f}   "
          f"DEMEANED {s['r_demeaned']:+.4f} (p={s['p']:.3f})")
    base = main.set_index('dataset').loc['fNIRS hBCI (v4)','r_demeaned']
    print(f"baseline demeaned {base:+.4f}  ->  combined {s['r_demeaned']:+.4f} "
          f"({s['r_demeaned']-base:+.4f})")

## 7. Deep encoder (EEGNet + TCN)

In [ ]:
import torch, torch.nn as nn
from scipy.signal import butter, filtfilt
torch.manual_seed(0)

def bandpower_seq(eeg, fs=EEG_FS, out_hz=20.0):
    N, C, T = eeg.shape; step = int(fs/out_hz); outs = []
    for lo, hi in BANDS_BASE:
        b, a = butter(4, [lo/(fs/2), min(hi/(fs/2), 0.99)], btype="band")
        env = filtfilt(b, a, eeg, axis=-1)**2
        outs.append(env[:, :, :(T//step)*step].reshape(N, C, T//step, step).mean(-1))
    x = np.log10(np.concatenate(outs, 1) + 1e-12).astype(np.float32)
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-6)

class TCN(nn.Module):
    def __init__(s, ch, dl):
        super().__init__()
        s.net = nn.Sequential(nn.Conv1d(ch,ch,3,padding=dl,dilation=dl), nn.BatchNorm1d(ch),
                              nn.ELU(), nn.Dropout(0.2),
                              nn.Conv1d(ch,ch,3,padding=dl,dilation=dl), nn.BatchNorm1d(ch), nn.ELU())
    def forward(s,x): return torch.relu(x + s.net(x))

class Enc(nn.Module):
    def __init__(s, in_ch, out_ch, out_len, latent=64):
        super().__init__()
        s.sp = nn.Sequential(nn.Conv1d(in_ch,128,1), nn.BatchNorm1d(128), nn.ELU(), nn.Dropout(0.25),
                             nn.Conv1d(128,latent,1), nn.BatchNorm1d(latent), nn.ELU())
        s.tcn = nn.Sequential(*[TCN(latent,d) for d in (1,2,4,8)])
        s.pool = nn.AdaptiveAvgPool1d(16)
        s.head = nn.Sequential(nn.Flatten(), nn.Linear(latent*16,512), nn.ELU(),
                               nn.Dropout(0.3), nn.Linear(512, out_ch*out_len))
        s.oc, s.ol = out_ch, out_len
    def forward(s,x): return s.head(s.pool(s.tcn(s.sp(x)))).view(-1, s.oc, s.ol)

def run_deep(d, epochs=60, bs=32, lr=1e-3, tag=""):
    Xtr, Xva = bandpower_seq(d["eeg_tr"]), bandpower_seq(d["eeg_va"])
    Ytr, Yva = d["hemo_tr"], d["hemo_va"]
    ys = Ytr.std() + 1e-8
    Xt, Yt = torch.tensor(Xtr), torch.tensor(Ytr/ys, dtype=torch.float32)
    Xv = torch.tensor(Xva).to(DEV)
    m = Enc(Xtr.shape[1], Ytr.shape[1], Ytr.shape[2]).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    n = len(Xt)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n); tot = 0.0
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = nn.functional.mse_loss(m(Xt[idx].to(DEV)), Yt[idx].to(DEV))
            loss.backward(); opt.step(); tot += loss.item()*len(idx)
        sch.step()
        if (ep+1) % 20 == 0: print(f"   {tag} ep{ep+1:3d} loss {tot/n:.4f}")
    m.eval()
    with torch.no_grad():
        return torch.cat([m(Xv[i:i+64]).cpu() for i in range(0,len(Xv),64)]).numpy()*ys

deep_rows = []
for k, d in DATA.items():
    print(f"training {k} ...")
    pred = run_deep(d, tag=k)
    Yva = d["hemo_va"]; sva = d["sid_va"]
    r_dm, null_dm, p_dm = perm_null(demean(pred.reshape(len(pred),-1), sva),
                                    demean(Yva.reshape(len(Yva),-1), sva))
    deep_rows.append({"dataset": k, "r_demeaned": r_dm, "null": null_dm, "p": p_dm})
    print(f"{k:20s} DEMEANED {r_dm:+.4f} (null {null_dm:+.4f}, p={p_dm:.3f})")
deep = pd.DataFrame(deep_rows)

## 8. Summary

In [ ]:
print("="*88); print("TARGET QUALITY"); print(qual.round(3).to_string(index=False))
print("\nRIDGE, CROSS-SUBJECT"); print(main.round(4).to_string(index=False))
print("\nDEEP ENCODER");        print(deep.round(4).to_string(index=False))
print("="*88)
out = Path("/kaggle/working" if ON_KAGGLE else ".")
qual.to_csv(out/"v4_target_quality.csv", index=False)
main.to_csv(out/"v4_ridge.csv", index=False)
deep.to_csv(out/"v4_deep.csv", index=False)
print("\nsaved CSVs to", out)
print("\nReference values from the local run:")
print("  fNIRS hBCI  demeaned 0.053 (p=0.005)  ->  0.071 with all three levers")
print("  fMRI fixed  no significant result at any setting")